# Tuần 3 — LLM Few-shot ICL + Aspect-aware RAG
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

Pipeline:
- **Tầng 2**: Gemini 1.5 Flash + Random few-shot (k = 2, 4, 8)
- **Tầng 3**: Gemini 1.5 Flash + Semantic RAG (vietnamese-sbert + FAISS)
- **So sánh**: Tầng 2 vs Tầng 3 vs PhoBERT (Tầng 1)

> Chạy theo thứ tự Cell 1 → Cell 8. Cần thêm `GEMINI_API_KEY` vào Kaggle Secrets trước.

In [ ]:
# ============================================================
# Cell 1 — Install dependencies
# ============================================================
import subprocess, sys

pkgs = [
    "google-genai",          # SDK mới thay thế google-generativeai
    "sentence-transformers",
    "faiss-cpu",
    "tabulate",
    "tqdm",
    "scikit-learn",
]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("✅ Dependencies installed")

# Kiểm tra import
from google import genai
from sentence_transformers import SentenceTransformer
print(f"  google-genai: OK")
print(f"  sentence-transformers: OK")
try:
    import faiss
    print(f"  faiss: OK")
except ImportError:
    print("  faiss: không có — sẽ dùng numpy fallback (chậm hơn một chút)")

In [ ]:
# ============================================================
# Cell 2 — Clone repo từ GitHub & Setup working directory
# ============================================================
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "master"
PROJECT_DIR = "/kaggle/working/absa-project"

# Clone (hoặc pull nếu đã có)
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print("✅ Clone xong")
else:
    print(f"Repo đã tồn tại — pulling latest...")
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

# Tạo thư mục cần thiết
for d in ["data", "outputs/results", "outputs/llm_cache", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

# Thêm Python paths
sys.path.insert(0, os.path.join(PROJECT_DIR, "code", "week3"))
sys.path.insert(0, os.path.join(PROJECT_DIR, "code", "week1"))

print("\nCấu trúc repo:")
!ls -la
!ls code/week1/ code/week3/

In [ ]:
# ============================================================
# Cell 3 — Download dataset VLSP 2018 + Preprocessing
# ============================================================
import pandas as pd, os

# Download raw data nếu chưa có
if not os.path.exists("data/train.csv"):
    print("Downloading VLSP 2018 Hotel dataset...")
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print("✅ Raw data downloaded")
else:
    print("✅ Raw data đã tồn tại")

for split in ["train", "dev", "test"]:
    df = pd.read_csv(f"data/{split}.csv")
    print(f"  {split}: {len(df)} rows")

# Preprocessing (dùng cache nếu có — không cần VnCoreNLP lại)
if os.path.exists("data/train_preprocessed.csv"):
    print("\n✅ Preprocessed cache đã có")
    s = pd.read_csv("data/train_preprocessed.csv").iloc[0]
    print(f"  Sample processed: {str(s.get('processed_review', 'N/A'))[:80]}")
else:
    print("\nChưa có preprocessed cache — chạy preprocessing...")
    # Cài VnCoreNLP
    !pip install -q py_vncorenlp underthesea
    import py_vncorenlp
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter

    vncorenlp_dir = os.path.join(os.getcwd(), "vncorenlp")
    if not os.path.exists(os.path.join(vncorenlp_dir, "models")):
        print("Downloading VnCoreNLP models...")
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)

    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f"data/{split}_preprocessed.csv")
        print(f"  ✅ {split}: {len(df)} rows")
    segmenter.close()
    print("✅ Preprocessing hoàn tất")

In [ ]:
# ============================================================
# Cell 4 — Chọn Gemini model
# ============================================================
# Đổi model ở đây, sẽ được dùng cho tất cả Cell 5 và Cell 6
import os

GEMINI_API_KEY = ""
from kaggle_secrets import UserSecretsClient
GEMINI_API_KEY = UserSecretsClient().get_secret("KEY_GEMINI_MINHVU")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
 # ← đổi model ở đây

from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)
print("Models hỗ trợ generateContent:")
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(f"  {m.name}")

In [ ]:
# ============================================================
# Cell 5 — Setup Gemini API Key
# ============================================================

GEMINI_MODEL = "models/gemini-2.5-flash" 
# Kiểm tra API key hợp lệ
from google import genai
client_test = genai.Client(api_key=GEMINI_API_KEY)
resp = client_test.models.generate_content(model=GEMINI_MODEL, contents="Reply with: OK")
print(f"✅ Gemini API OK — model: {GEMINI_MODEL} | response: {resp.text.strip()}")

In [ ]:
# ============================================================
# Cell 6 — Tầng 2: Few-shot ICL (Gemini, k = 2, 4, 8)
# ============================================================
import pandas as pd
from icl_predictor import run_icl_ablation

train_df = pd.read_csv("data/train_preprocessed.csv")
test_df  = pd.read_csv("data/test_preprocessed.csv")

print(f"Train: {len(train_df)} | Test: {len(test_df)}")

# ─── Config ───────────────────────────────────────────────────
MAX_SAMPLES = None     # None = full 600 | 50 = test nhanh
K_VALUES    = [2, 4, 8]
SLEEP_SEC   = 7.0      # Free tier: 10 RPM → cần ≥6s. Trả tiền: đặt 1.0
# ──────────────────────────────────────────────────────────────

print(f"\n{'='*55}")
print(f"Tầng 2: Few-shot ICL — {GEMINI_MODEL}")
print(f"k values: {K_VALUES} | samples: {MAX_SAMPLES or len(test_df)} | sleep: {SLEEP_SEC}s")
print(f"Ước tính thời gian: ~{(MAX_SAMPLES or len(test_df)) * len(K_VALUES) * SLEEP_SEC / 60:.0f} phút")
print(f"{'='*55}\n")

tier2_results = run_icl_ablation(
    test_df=test_df,
    train_df=train_df,
    providers=["gemini"],
    k_values=K_VALUES,
    api_keys={"gemini": GEMINI_API_KEY},
    models={"gemini": GEMINI_MODEL},
    results_dir="outputs/results",
    max_samples=MAX_SAMPLES,
    sleep_sec=SLEEP_SEC,
)

print("\n" + "="*55)
print("TẦNG 2 — KẾT QUẢ")
print("="*55)
for exp, m in sorted(tier2_results.items()):
    print(f"  {exp}: Combined F1 = {m['macro_combined_f1']:.4f} "
          f"| ACD = {m['macro_acd_f1']:.4f} "
          f"| SPC = {m['macro_spc_f1']:.4f}")

In [ ]:
# ============================================================
# Cell 6 — Tầng 3: Aspect-aware RAG (Gemini, k = 2, 4, 8)
# ============================================================
import pandas as pd
from rag_predictor import run_rag_ablation

train_df = pd.read_csv("data/train_preprocessed.csv")
test_df  = pd.read_csv("data/test_preprocessed.csv")

# ─── Config (GEMINI_MODEL và SLEEP_SEC kế thừa từ Cell 5) ─────
MAX_SAMPLES = None   # Đặt 50 để test nhanh
K_VALUES    = [2, 4, 8]
# ──────────────────────────────────────────────────────────────

print(f"{'='*55}")
print(f"Tầng 3: Aspect-aware RAG — {GEMINI_MODEL}")
print(f"k values: {K_VALUES} | samples: {MAX_SAMPLES or len(test_df)} | sleep: {SLEEP_SEC}s")
print(f"Ước tính thời gian: ~{(MAX_SAMPLES or len(test_df)) * len(K_VALUES) * SLEEP_SEC / 60:.0f} phút")
print(f"{'='*55}\n")

tier3_results = run_rag_ablation(
    test_df=test_df,
    train_df=train_df,
    providers=["gemini"],
    k_values=K_VALUES,
    api_keys={"gemini": GEMINI_API_KEY},
    models={"gemini": GEMINI_MODEL},
    results_dir="outputs/results",
    max_samples=MAX_SAMPLES,
    sleep_sec=SLEEP_SEC,
)

print("\n" + "="*55)
print("TẦNG 3 — KẾT QUẢ")
print("="*55)
for exp, m in sorted(tier3_results.items()):
    print(f"  {exp}: Combined F1 = {m['macro_combined_f1']:.4f} "
          f"| ACD = {m['macro_acd_f1']:.4f} "
          f"| SPC = {m['macro_spc_f1']:.4f}")

In [ ]:
# ============================================================
# Cell 7 — So sánh kết quả & Tạo báo cáo Markdown
# ============================================================
import json, os
import pandas as pd
from compare_results import generate_comparison_table, load_all_results, WEEK2_RESULTS

# Load tất cả kết quả tier2 + tier3
all_results = load_all_results("outputs/results")

# ─── In bảng tóm tắt nhanh ───────────────────────────────────
print(f"{'='*65}")
print(f"{'Method':<35} {'ACD F1':>8} {'SPC F1':>8} {'Combined':>10}")
print(f"{'─'*65}")

# PhoBERT baseline (từ week2)
for name, m in WEEK2_RESULTS.items():
    print(f"  [Tầng 1] {name:<26} {m['acd_f1']:>8.4f} {m['spc_f1']:>8.4f} {m['combined_f1']:>10.4f}")

print(f"{'─'*65}")

# LLM results sorted by combined F1
sorted_results = sorted(all_results.items(), key=lambda x: -x[1]["combined_f1"])
for name, m in sorted_results:
    tier = "Tầng 2" if "tier2" in name else "Tầng 3"
    print(f"  [{tier}] {name:<30} {m['acd_f1']:>8.4f} {m['spc_f1']:>8.4f} {m['combined_f1']:>10.4f}")

print(f"{'─'*65}")
print(f"  [SOTA]  phobert_huynh2022              {'0.8255':>8} {'—':>8} {'0.7732':>10}")
print(f"{'='*65}")

# ─── RAG vs ICL so sánh ───────────────────────────────────────
print("\n=== RAG vs ICL (same k, same model) ===")
for k in [2, 4, 8]:
    icl_key = f"tier2_gemini_k{k}"
    rag_key = f"tier3_gemini_k{k}"
    if icl_key in all_results and rag_key in all_results:
        icl_f1 = all_results[icl_key]["combined_f1"]
        rag_f1 = all_results[rag_key]["combined_f1"]
        gain   = rag_f1 - icl_f1
        symbol = "✅" if gain > 0 else "❌"
        print(f"  k={k}: ICL={icl_f1:.4f} → RAG={rag_f1:.4f} ({gain*100:+.2f}%) {symbol}")

# ─── Tạo file báo cáo Markdown ───────────────────────────────
report = generate_comparison_table(
    results_dir="outputs/results",
    save_path="outputs/results/week3_comparison.md",
)
print("\n✅ Báo cáo đã lưu: outputs/results/week3_comparison.md")

In [ ]:
# ============================================================
# Cell 8 — Download kết quả
# ============================================================
import os, shutil, json

# In nội dung báo cáo
report_path = "outputs/results/week3_comparison.md"
if os.path.exists(report_path):
    print(open(report_path, encoding="utf-8").read())

# Liệt kê files kết quả tuần 3
print("\n=== Kết quả đã tạo ===")
for f in sorted(os.listdir("outputs/results")):
    if "tier2" in f or "tier3" in f or "week3" in f:
        path = f"outputs/results/{f}"
        size = os.path.getsize(path)
        print(f"  {path} ({size/1024:.1f} KB)")

# Tạo zip để download
zip_path = "/kaggle/working/week3_results"
shutil.make_archive(zip_path, "zip", "outputs/results")
print(f"\n✅ Zip: {zip_path}.zip")
print("   → Kaggle: Output panel bên phải → Download")

# In số liệu key
print("\n=== Số liệu key (cho báo cáo) ===")
all_results = {}
for fname in os.listdir("outputs/results"):
    if fname.endswith("_metrics.json") and ("tier2" in fname or "tier3" in fname):
        data = json.load(open(f"outputs/results/{fname}"))
        key = fname.replace("_metrics.json", "")
        all_results[key] = data

if all_results:
    best = max(all_results.items(), key=lambda x: x[1].get("macro_combined_f1", 0))
    print(f"  Best method:     {best[0]}")
    print(f"  Best ACD F1:     {best[1]['macro_acd_f1']:.4f}")
    print(f"  Best SPC F1:     {best[1]['macro_spc_f1']:.4f}")
    print(f"  Best Combined:   {best[1]['macro_combined_f1']:.4f}")